In [1]:
import torch
import torch.nn as nn

In [2]:
import zipfile

zip_path = r"C:\Users\yapri\Downloads\archive (2).zip"
extract_path = r"C:\Users\yapri\Downloads\catsvsdogs_data"

with zipfile.ZipFile(zip_path, "r") as zip_r:
    zip_r.extractall(extract_path)

In [3]:
from torchvision import datasets,transforms
device=torch.device("cuda")
train_path=r"C:\Users\yapri\Downloads\catsvsdogs_data\catsvsdogs\train"
test_path=r"C:\Users\yapri\Downloads\catsvsdogs_data\catsvsdogs\test"
train_transform=transforms.Compose([transforms.Resize((256,256)),
                                    transforms.RandomHorizontalFlip(p=0.5),
                                    transforms.RandomRotation(degrees=15),
                                    transforms.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2),
                                    transforms.ToTensor()])
test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()])
train_dataset=datasets.ImageFolder(root=train_path,transform=train_transform)
test_dataset=datasets.ImageFolder(root=test_path,transform=test_transform)

In [4]:
print("Train images:", len(train_dataset))
print("Test images:", len(test_dataset))
print("Classes:", train_dataset.classes)
print("Class mapping:", train_dataset.class_to_idx)

Train images: 20000
Test images: 5000
Classes: ['cats', 'dogs']
Class mapping: {'cats': 0, 'dogs': 1}


In [5]:
from torch.utils.data import  Dataset, DataLoader
train_loader=DataLoader(train_dataset,
                        batch_size=32,
                        shuffle=True,
                        pin_memory=True,
                        num_workers=2)
test_loader = DataLoader(test_dataset,
                         batch_size=32,
                         shuffle=False,
                         num_workers=2,
                         pin_memory=True)

In [6]:
# to check data is properly fetched
images, labels = next(iter(train_loader))
print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Labels:", labels[:10])
images = images.to(device, non_blocking=True)
labels = labels.to(device, non_blocking=True)
print("Image device:", images.device)
print("Label device:", labels.device)

Images shape: torch.Size([32, 3, 256, 256])
Labels shape: torch.Size([32])
Labels: tensor([0, 0, 1, 1, 0, 1, 0, 1, 1, 1])
Image device: cuda:0
Label device: cuda:0


In [7]:
class catdogcnn(nn.Module):
    def __init__(self,channel):
        super().__init__()
        self.feature=nn.Sequential(nn.Conv2d(channel,32,kernel_size=3,padding=1),
                                   nn.ReLU(),
                                   nn.MaxPool2d(kernel_size=2,stride=2),
                                   nn.Conv2d(32,64,kernel_size=3,padding=1),
                                   nn.ReLU(),
                                   nn.MaxPool2d(kernel_size=2,stride=2),
                                   nn.Conv2d(64,128,kernel_size=3,padding=1),
                                   nn.ReLU(),
                                   nn.MaxPool2d(kernel_size=2,stride=2))
        self.classifier=nn.Sequential(nn.Flatten(),nn.Linear(32*32*128,128),
                                     nn.ReLU(),
                                     nn.Dropout(0.3),
                                     nn.Linear(128,1))
    def forward(self,x):
            x=self.feature(x)
            x=self.classifier(x)
            return x       

In [8]:
model=catdogcnn(3)
model=model.to(device)

In [9]:
epochs=20
loss_fun=nn.BCEWithLogitsLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

In [11]:
for epoch in range(epochs):
    for batch_feature,batch_label in train_loader:
        batch_feature,batch_label=batch_feature.to(device),batch_label.to(device)
        y_pred=model.forward(batch_feature)
        loss=loss_fun(y_pred,batch_label.view(-1,1).float())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(f"epoch:{epoch},loss:{loss.item()}")


epoch:0,loss:0.3539646863937378
epoch:0,loss:0.20422518253326416
epoch:0,loss:0.21902450919151306
epoch:0,loss:0.44627195596694946
epoch:0,loss:0.3346126675605774
epoch:0,loss:0.16574852168560028
epoch:0,loss:0.215196430683136
epoch:0,loss:0.15316523611545563
epoch:0,loss:0.3509144186973572
epoch:0,loss:0.3135213255882263
epoch:0,loss:0.25748303532600403
epoch:0,loss:0.2283240109682083
epoch:0,loss:0.1988990604877472
epoch:0,loss:0.24790692329406738
epoch:0,loss:0.3828548789024353
epoch:0,loss:0.14655259251594543
epoch:0,loss:0.2395671010017395
epoch:0,loss:0.3739582300186157
epoch:0,loss:0.14833185076713562
epoch:0,loss:0.29175370931625366
epoch:0,loss:0.14233386516571045
epoch:0,loss:0.3058320879936218
epoch:0,loss:0.2750939726829529
epoch:0,loss:0.2120549976825714
epoch:0,loss:0.23185256123542786
epoch:0,loss:0.2808496654033661
epoch:0,loss:0.255898118019104
epoch:0,loss:0.24735897779464722
epoch:0,loss:0.21644562482833862
epoch:0,loss:0.1764216125011444
epoch:0,loss:0.2312882840633

In [55]:

model.eval()
for epoch in range(10):
    val_loss=0
    with torch.no_grad():
        for batch_feature,batch_label in test_loader:
            batch_feature,batch_label=batch_feature.to(device),batch_label.to(device)
            output=model.forward(batch_feature)
            loss=loss_fun(output,batch_label.view(-1,1).float())
            val_loss+=loss.item()
    avg_val_loss=val_loss/len(test_loader)
    print(f"Epoch {epoch}: Validation Loss = {avg_val_loss:.4f}")

Epoch 0: Validation Loss = 0.3630
Epoch 1: Validation Loss = 0.3630
Epoch 2: Validation Loss = 0.3630
Epoch 3: Validation Loss = 0.3630
Epoch 4: Validation Loss = 0.3630
Epoch 5: Validation Loss = 0.3630
Epoch 6: Validation Loss = 0.3630
Epoch 7: Validation Loss = 0.3630
Epoch 8: Validation Loss = 0.3630
Epoch 9: Validation Loss = 0.3630


In [12]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch_feature, batch_label in test_loader:
        batch_feature = batch_feature.to(device)
        batch_label = batch_label.to(device)
        output = model(batch_feature)
        prediction = (torch.sigmoid(output) >= 0.5).long()
        correct += (prediction.view(-1) == batch_label).sum().item()
        total += batch_label.size(0)
accuracy = correct / total
print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 88.86%


In [26]:
 from PIL import Image 
import torch 
image_path=r"C:\Users\yapri\Downloads\cat_1.webp"
image=Image.open(image_path).convert("RGB")
image=test_transform(image)
image=image.unsqueeze(0)
image=image.to(device)
model.eval()
with torch.no_grad():
    output=model.forward(image)
    probability=torch.sigmoid(output).item()
    if probability >=0.7:
        prediction="dog"
    elif probability<=0.2:
        prediction="cat"
    else:
        prediction="not confident"
print("Prediction:", prediction)
print(f"Dog Probability: {probability * 100:.2f}%")
print(f"Cat Probability: {(1 - probability) * 100:.2f}%")    

Prediction: cat
Dog Probability: 1.14%
Cat Probability: 98.86%


In [27]:
torch.save(model.state_dict(), "cat_dog_cnn.pth")